In [ ]:
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd

from sklearn.linear_model import Ridge, RidgeCV, ElasticNet,ElasticNetCV, LassoCV, lasso_path, enet_path
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis as QDA
from sklearn.neighbors import KNeighborsClassifier


import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import LeaveOneOut, cross_validate, KFold

from ISLP.models import (ModelSpec as MS, summarize , poly)

from sklearn.decomposition import PCA
from sklearn.model_selection import LeaveOneOut, cross_validate, KFold

from sklearn.cluster import KMeans

# PCA

## Iris Data
From https://archive.ics.uci.edu/dataset/53/iris

In [ ]:
iris = pd.read_csv('../data/iris.data', header=None, names=['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species'])
iris['species'] = iris['species'].astype('category')
iris.head()

In [ ]:
pca_pipe = make_pipeline(StandardScaler(), PCA(n_components=2))
iris_pca_scaled = pca_pipe.fit_transform(iris.drop(columns='species'))

In [ ]:
pca_pipe[1].components_, pca_pipe[1].explained_variance_ratio_

In [ ]:
fig, axes = plt.subplots()
for species, color in zip(iris['species'].cat.categories, ['C0', 'C1', 'C2']):
    species_idx = iris['species'] == species
    axes.scatter(
        iris_pca_scaled[species_idx, 0], iris_pca_scaled[species_idx, 1],
        s=10, alpha=0.5, label=species, color=color
    )

# Add loading vectors for the original features in PC space
loadings = pca_pipe[1].components_.T
feature_names = iris.drop(columns='species').columns
score_scale = np.max(np.abs(iris_pca_scaled[:, :2]), axis=0)
arrow_scale = 0.8

for i, feature in enumerate(feature_names):
    vec = loadings[i, :] * score_scale * arrow_scale
    axes.arrow(
        0, 0, vec[0], vec[1],
        color='black', width=0.015, alpha=0.7,
        head_width=0.12, head_length=0.16, length_includes_head=True
    )
    axes.text(vec[0] * 1.1, vec[1] * 1.1, feature, color='black', fontsize=9)

axes.set_xlabel("PC1")
axes.set_ylabel("PC2")
axes.set_title("PCA Projection of Iris Data with Loading Vectors")  
axes.legend()

# K-Means

## Three Clusters

In [ ]:


np.random.seed(123)

n = 10**4
weights = np.array([0.25, 0.5, 0.25])

means = np.array([
    [-1.0,  -2.0],
    [ -1.0,  1.0],
    [ 2.0, 2.0]
])

covs = np.array([
    [[1.0,  0.3],
     [0.3,  1.0]],
    [[1.0,  0.0],
     [0.0,  1.0]],
    [[1.0,  0.0],
     [0.0,  4.0]]
])

# Sample classes
z = np.random.choice(3, size=n, p=weights)

# Sample positions
x = np.zeros((n, 2))
for i in range(n):
    x[i,:] =np.random.multivariate_normal(means[z[i]], covs[z[i]])

# store in a data frame
df = pd.DataFrame(x, columns=['x1', 'x2'])
df['class'] = z
df['class'] = df['class'].astype('category')
fig, axes = plt.subplots()
for k, g in df.groupby("class"):
    g.plot.scatter("x1", "x2", s=10, alpha=0.5, label=f"Class {k}", color=f"C{k}", ax=axes)
axes.set_title("3-Component 2D Gaussian Mixture")
axes.set_xlabel(r"$x_1$")
axes.set_ylabel(r"$x_2$")
axes.legend()

In [ ]:
kmeans = KMeans(n_clusters=3,
                random_state=2,
                n_init=20).fit(df)


In [ ]:
kmeans

In [ ]:
kmeans.labels_

In [ ]:
fig, ax = plt.subplots()
df.plot.scatter("x1", "x2", s=10, alpha=0.5, ax=ax, c=kmeans.labels_, cmap='viridis', colorbar=False)
ax.set_title("K-Means Clustering Results with K=3");    


## Iris Data

In [ ]:
kmeans_iris = KMeans(n_clusters=3,
                random_state=318,
                n_init=50).fit(iris.drop(columns='species'))


In [ ]:
kmeans_iris.labels_

In [ ]:
fig, ax = plt.subplots()
ax.scatter(iris_pca_scaled[:, 0], iris_pca_scaled[:, 1],s=10, alpha=0.5,  c=kmeans_iris.labels_, cmap='viridis')


In [ ]:
fig, axes = plt.subplots(1,2, figsize=(12, 5))
for species, color in zip(iris['species'].cat.categories, ['C0', 'C1', 'C2']):
    species_idx = iris['species'] == species
    axes[0].scatter(
        iris_pca_scaled[species_idx, 0], iris_pca_scaled[species_idx, 1],
        s=10, alpha=0.5, label=species, color=color
    )
axes[0].set_title("PCA Projection Colored by True Species")
axes[1].scatter(iris_pca_scaled[:, 0], iris_pca_scaled[:, 1],s=10, alpha=0.5,  c=kmeans_iris.labels_, cmap='viridis')
axes[1].set_title("PCA Projection Colored by K-Means Clusters")



# Using PCA in a Classifier

In [ ]:
iris['z1'] =iris_pca_scaled[:,0]
iris['z2'] =iris_pca_scaled[:,1]

In [ ]:
iris.head()

In [ ]:
lda = LDA()
qda = QDA()
knn3 = KNeighborsClassifier(n_neighbors=3)
knn10 = KNeighborsClassifier(n_neighbors=10)



kfold_cv = KFold(n_splits=10, shuffle=True, random_state=318)

cv_results_lda = cross_validate(lda, iris[['z1', 'z2']], iris['species'], cv=kfold_cv, return_train_score=True,scoring='accuracy')
cv_results_qda = cross_validate(qda, iris[['z1', 'z2']], iris['species'], cv=kfold_cv, return_train_score=True,scoring='accuracy')
cv_results_knn3 = cross_validate(knn3, iris[['z1', 'z2']], iris['species'], cv=kfold_cv, return_train_score=True,scoring='accuracy')
cv_results_knn10 = cross_validate(knn10, iris[['z1', 'z2']], iris['species'], cv=kfold_cv, return_train_score=True,scoring='accuracy')


In [ ]:
print(1- cv_results_lda['train_score'].mean(), 1- cv_results_lda['test_score'].mean())
print(1- cv_results_qda['train_score'].mean(), 1- cv_results_qda['test_score'].mean())
print(1- cv_results_knn3['train_score'].mean(), 1- cv_results_knn3['test_score'].mean())
print(1- cv_results_knn10['train_score'].mean(), 1- cv_results_knn10['test_score'].mean())